# Prepare 2012 Kauring AirGrav ASEG-GDF2 data

**__See EastVic Data for latest organisation of Prepare tutorials__**

This short tutorial demonstrates the preparation of ASEG-GDF2 data for QC review.

We need all the data in HDF5 geoWhizz format because all the QC functions expect that format. (More on the geoWhizz format elsewhere in the __galileoQC__ documentation.)

For this example we are using the data from the Kauring airborne gravity survey flown by SGL in 2012.

This step is reasonably straightforward and once complete, the QC of the data should proceed quickly and smoothly, and the outputs will be clearly documented so that the JupyterLab notebook can serve as the QC report.

___

**Import modules and set filenames**

In [ ]:
from pathlib import Path
import galileoQC  as qc

In [ ]:
dx = Path(r'./Kauring_grv/GRAV.DAT')

___

**2012 Kauring Gravimeter Survey**

`asegToHDF` does all the work. You must specify the channel containing the line number so that it can organise the data by flight-line. Optionally, you can also supply the channels containing the flight number, and date (not present in the 2012 Kauring data), and a list of channels you do not want copied across to the geowhizz HDF file.

In [ ]:
# asegToHdf fails if the file exists so delete it if it does. CHECK
if dx.with_suffix('.hdf5').exists(): 
    dx.with_suffix('.hdf5').unlink()

Here we omit many channels to keep the volume of data in the repository reasonably sized.

In [ ]:
dh = qc.asegToHDF(Path(dx),
            lineChannel='LINE',
            flightChannel='FLIGHT',
            omitChannels=['SECOND', 'DAYOFYEAR', 'YEAR',
            'ALTTER', 'BAREEARTHTER', 'DEMTER', 'FZ', 'AZ',
            'GRVRAW', 'LATCOR', 'EOTCOR', 'FACOR', 'GRVFRA100S', 'INTCOR',
            'GRVFRAL100S', 'FTBCOR_267', 'GRVFAL0600M', 'GRVFAL1000M',
            'FVDBGL0600M_267', 'FVDBGL1000M_267', 'GNDGRV_FA_UPWARD',
            'GRVBGL0600M_267', 'GRVBGL1000M_267', 'GNDGRV_BG_UPWARD'])


Once the file is created, we can add in the project and block name. These are used in plot titles, keeping the report informative and clear.


In [ ]:
block_name = 'Kauring Western Australia SGL 2012'
project_name = 'Kauring Airborne Gravity'
qc.updateProject(dh, acquirer='SGL', projectName=project_name, blockID=block_name)

The CoordFrame metadata provides the names of default channels, and related coordinate information, for the QC functions.

In [ ]:
qc.updateCoordFrame(dh, 
                    lat='LATITUDE', 
                    lon='LONGITUD', 
                    x='EASTING', 
                    y='NORTHING', 
                    time='UTCSECOND', 
                    alt='GPSZ', 
                    geoDatum='WGS84', 
                    htDatum='GRS80', 
                    projection='UTM', 
                    utmz='50')

The `line_type` tells `updateLineAttributes` how to extract the planned line number, the segment number, and the repeat number from each reported survey line number. It also allows it to classify flight-lines as traverse or control lines. This information is used by various QC functions.

More details on Line Types can be found elsewhere in the documentation.

In [ ]:
qc.updateLineAttributes(dh, line_type='SGL_Kauring')

Now we can report the contents of our new geowhizz file.

In [ ]:
qc.reportWhizz(dh)

The units, and the description, if it exists and is formatted according to the standard, are copied from the DFN file.

In [ ]:
qc.reportChannels(dh, verbose=True)

In [ ]:
qc.reportSampling(dh)

We can check that the traverse and control lines have been correctly classified. We have not asked for a classification on the basis of odd or even line-numbering.

In [ ]:
qc.reportWhizz(dh, line='100100')

___

**Make a flight-line map**

In [ ]:
qc.linesMap([dh], whizzPlanFile=dh)